# 02 - TF-IDF multivista + MLP full-batch

Notebook generado automáticamente para el proyecto de predicción de década.

## 0. Instalación y configuración

In [ ]:
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
%pip install pandas numpy scipy scikit-learn matplotlib tqdm

In [ ]:
# =========================
# Imports generales
# =========================

import os
import gc
import math
import time
import random
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp

# =========================
# Visualización
# =========================

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# =========================
# Scikit-learn
# =========================

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    classification_report,
    confusion_matrix,
)

# =========================
# PyTorch
# =========================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


In [ ]:
# =========================
# Configuración general para GPU
# =========================

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("CONFIGURACIÓN DEL ENTORNO")
print("=" * 70)
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("CUDA PyTorch:", torch.version.cuda)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria GPU:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
    print("Memoria asignada:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
    print("Memoria reservada:", round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")

print("=" * 70)


In [ ]:
# =========================
# Limpieza segura de memoria
# =========================

def clean_memory(verbose=True):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        if verbose:
            print("Memoria asignada:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
            print("Memoria reservada:", round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")

clean_memory()


## 1. Configuración del experimento

In [ ]:
# =========================
# Configuración del experimento: TF-IDF multivista + MLP full-batch
# =========================

TRAIN_PATH = "train.csv"
TEXT_COL = "text"
LABEL_COL = "decade"

CHAR_NGRAM_RANGE = (2, 6)
CHAR_WB_NGRAM_RANGE = (3, 6)
WORD_NGRAM_RANGE = (1, 2)

CHAR_MAX_FEATURES = 220_000
CHAR_WB_MAX_FEATURES = 160_000
WORD_MAX_FEATURES = 100_000

MIN_DF = 2
MAX_DF = 1.0

# Full-batch usa mucha memoria; si falla, baja hidden_1.
HIDDEN_1 = 768
HIDDEN_2 = 384
HIDDEN_3 = 192
DROPOUT = 0.35

EPOCHS = 30
PATIENCE = 2

# Batch para predicción/evaluación solamente. El entrenamiento usa full batch.
BATCH_SIZE = 512 if DEVICE.type == "cuda" else 128

LR = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
GRAD_CLIP = 1.0

CHECKPOINT_DIR = "checkpoints_02_tfidf_multiview_mlp_fullbatch"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best.pt")
LAST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "last.pt")

print("Entrenamiento full-batch activado.")
print("Checkpoints:", CHECKPOINT_DIR)


## 2. Datos y etiquetas

In [ ]:
# =========================
# Carga de datos y mapeo de etiquetas
# =========================

df = pd.read_csv(TRAIN_PATH)

assert TEXT_COL in df.columns, f"No existe columna '{TEXT_COL}'"
assert LABEL_COL in df.columns, f"No existe columna '{LABEL_COL}'"

df = df[[TEXT_COL, LABEL_COL]].copy()
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)

decades = sorted(df[LABEL_COL].unique())
decade_to_idx = {decade: idx for idx, decade in enumerate(decades)}
idx_to_decade = {idx: decade for decade, idx in decade_to_idx.items()}

df["label_idx"] = df[LABEL_COL].map(decade_to_idx)

NUM_CLASSES = len(decades)

print("Shape:", df.shape)
print("Número de clases:", NUM_CLASSES)
print("Década mínima:", min(decades))
print("Década máxima:", max(decades))
print("Clases:", decades)

# Chequeos explícitos de etiquetas extremas
assert min(decades) == 150, "La década mínima no es 150. Revisa el dataset."
assert max(decades) == 188, "La década máxima no es 188. Revisa el dataset."
assert NUM_CLASSES == 39, f"Se esperaban 39 clases, pero hay {NUM_CLASSES}."

display(df.head())
display(df[LABEL_COL].value_counts().sort_index())


In [ ]:
# =========================
# Split train / validation / test
# =========================

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label_idx"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_idx"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

# Verificar que las clases extremas estén en todos los splits
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    present = set(part[LABEL_COL].unique())
    print(name, "contiene 150:", 150 in present, "| contiene 188:", 188 in present)


## 3. TF-IDF multivista

In [ ]:
# =========================
# Vectorizadores TF-IDF multivista
# =========================

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=CHAR_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    dtype=np.float32
)

char_wb_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=CHAR_WB_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=CHAR_WB_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    dtype=np.float32
)

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=WORD_NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=WORD_MAX_FEATURES,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    token_pattern=r"(?u)\b\w+\b",
    dtype=np.float32
)

train_texts = train_df[TEXT_COL].tolist()
val_texts = val_df[TEXT_COL].tolist()
test_texts = test_df[TEXT_COL].tolist()

print("Ajustando char TF-IDF...")
X_train_char = char_vectorizer.fit_transform(train_texts)
X_val_char = char_vectorizer.transform(val_texts)
X_test_char = char_vectorizer.transform(test_texts)

print("Ajustando char_wb TF-IDF...")
X_train_char_wb = char_wb_vectorizer.fit_transform(train_texts)
X_val_char_wb = char_wb_vectorizer.transform(val_texts)
X_test_char_wb = char_wb_vectorizer.transform(test_texts)

print("Ajustando word TF-IDF...")
X_train_word = word_vectorizer.fit_transform(train_texts)
X_val_word = word_vectorizer.transform(val_texts)
X_test_word = word_vectorizer.transform(test_texts)

X_train = sp.hstack([X_train_char, X_train_char_wb, X_train_word], format="csr", dtype=np.float32)
X_val = sp.hstack([X_val_char, X_val_char_wb, X_val_word], format="csr", dtype=np.float32)
X_test = sp.hstack([X_test_char, X_test_char_wb, X_test_word], format="csr", dtype=np.float32)

y_train = train_df["label_idx"].values.astype(np.int64)
y_val = val_df["label_idx"].values.astype(np.int64)
y_test = test_df["label_idx"].values.astype(np.int64)

INPUT_DIM = X_train.shape[1]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("Input dim:", INPUT_DIM)
print("Sparsity train:", 1.0 - (X_train.nnz / (X_train.shape[0] * X_train.shape[1])))


In [ ]:
# =========================
# Dataset sparse por índices
# =========================

class SparseIndexDataset(Dataset):
    def __init__(self, labels):
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {"idx": idx, "label": self.labels[idx]}


def sparse_collate_fn(batch):
    indices = torch.tensor([item["idx"] for item in batch], dtype=torch.long)
    labels = torch.stack([item["label"] for item in batch])
    return {"indices": indices, "labels": labels}


class EvalSparseIndexDataset(Dataset):
    def __init__(self, indices):
        self.indices = torch.tensor(indices, dtype=torch.long)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return {"idx": self.indices[idx]}


def eval_sparse_collate_fn(batch):
    indices = torch.stack([item["idx"] for item in batch])
    return {"indices": indices}


def csr_batch_to_torch_sparse(X_csr_batch, device=DEVICE):
    X_coo = X_csr_batch.tocoo()
    indices = torch.tensor(np.vstack((X_coo.row, X_coo.col)), dtype=torch.long)
    values = torch.tensor(X_coo.data, dtype=torch.float32)
    shape = torch.Size(X_coo.shape)
    return torch.sparse_coo_tensor(indices, values, shape, dtype=torch.float32).coalesce().to(device)


train_dataset = SparseIndexDataset(y_train)
val_dataset = SparseIndexDataset(y_val)
test_dataset = SparseIndexDataset(y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=sparse_collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=sparse_collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=sparse_collate_fn, num_workers=0)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))


## 4. Modelo

In [ ]:
# =========================
# MLP potente para TF-IDF sparse
# =========================

class SparseTfidfMLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_1=HIDDEN_1, hidden_2=HIDDEN_2, hidden_3=HIDDEN_3, dropout=DROPOUT):
        super().__init__()

        # Primera capa manual para soportar input sparse
        self.fc1_weight = nn.Parameter(torch.empty(hidden_1, input_dim))
        self.fc1_bias = nn.Parameter(torch.zeros(hidden_1))

        self.norm1 = nn.LayerNorm(hidden_1)
        self.dropout1 = nn.Dropout(dropout)

        self.fc2 = nn.Linear(hidden_1, hidden_2)
        self.norm2 = nn.LayerNorm(hidden_2)
        self.dropout2 = nn.Dropout(dropout)

        self.fc3 = nn.Linear(hidden_2, hidden_3)
        self.norm3 = nn.LayerNorm(hidden_3)
        self.dropout3 = nn.Dropout(dropout)

        self.out = nn.Linear(hidden_3, num_classes)

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.fc1_weight)
        nn.init.zeros_(self.fc1_bias)
        for module in [self.fc2, self.fc3, self.out]:
            nn.init.xavier_uniform_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x_sparse):
        x = torch.sparse.mm(x_sparse, self.fc1_weight.t()) + self.fc1_bias
        x = self.norm1(x)
        x = F.gelu(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.norm2(x)
        x = F.gelu(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        x = self.norm3(x)
        x = F.gelu(x)
        x = self.dropout3(x)

        return self.out(x)


model = SparseTfidfMLP(INPUT_DIM, NUM_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Total parameters: {total_params:,}")


In [ ]:
# =========================
# Métricas
# =========================

idx_to_decade_tensor = torch.tensor([idx_to_decade[i] for i in range(NUM_CLASSES)], dtype=torch.long, device=DEVICE)

def compute_metrics_from_logits(logits, labels):
    preds = torch.argmax(logits, dim=1)
    pred_decades = idx_to_decade_tensor[preds]
    true_decades = idx_to_decade_tensor[labels]
    abs_error = torch.abs(pred_decades - true_decades)

    return {
        "acc": (preds == labels).float().mean().item(),
        "acc_pm1": (abs_error <= 1).float().mean().item(),
        "acc_pm2": (abs_error <= 2).float().mean().item(),
        "mae": abs_error.float().mean().item(),
    }


In [ ]:
# =========================
# Funciones full-batch
# =========================

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1
)

def fullbatch_to_device(X_matrix, y_array):
    X_tensor = csr_batch_to_torch_sparse(X_matrix, device=DEVICE)
    y_tensor = torch.tensor(y_array, dtype=torch.long, device=DEVICE)
    return X_tensor, y_tensor


def train_one_epoch_fullbatch(model, X_matrix, y_array):
    model.train()
    X_tensor, labels = fullbatch_to_device(X_matrix, y_array)

    optimizer.zero_grad(set_to_none=True)
    logits = model(X_tensor)
    loss = criterion(logits, labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()

    metrics = compute_metrics_from_logits(logits.detach(), labels.detach())
    metrics["loss"] = loss.item()

    del X_tensor, labels, logits, loss
    clean_memory(verbose=False)

    return metrics


@torch.no_grad()
def evaluate_fullbatch(model, X_matrix, y_array, desc="Evaluating"):
    model.eval()
    X_tensor, labels = fullbatch_to_device(X_matrix, y_array)

    logits = model(X_tensor)
    loss = criterion(logits, labels)

    metrics = compute_metrics_from_logits(logits.detach(), labels.detach())
    metrics["loss"] = loss.item()

    del X_tensor, labels, logits, loss
    clean_memory(verbose=False)

    return metrics


# Para predicción/submission usamos mini-batches de inferencia, no afecta la condición de entrenamiento full-batch.
@torch.no_grad()
def predict_sparse(model, X_matrix, batch_size=BATCH_SIZE):
    model.eval()
    indices = np.arange(X_matrix.shape[0])
    ds = EvalSparseIndexDataset(indices)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=eval_sparse_collate_fn, num_workers=0)

    all_preds = []
    all_logits = []

    for batch in tqdm(loader, desc="Predicting", leave=False):
        batch_indices = batch["indices"].numpy()
        X_batch = csr_batch_to_torch_sparse(X_matrix[batch_indices], device=DEVICE)
        logits = model(X_batch)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_logits.append(logits.detach().cpu())

    return np.array(all_preds), torch.cat(all_logits, dim=0)


## 5. Entrenamiento full-batch

In [ ]:
# =========================
# Loop principal full-batch: prioriza accuracy
# =========================

history = []
best_val_acc = -1.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 70)

    train_metrics = train_one_epoch_fullbatch(model, X_train, y_train)
    val_metrics = evaluate_fullbatch(model, X_val, y_val, desc="Validation")
    scheduler.step(val_metrics["acc"])

    row = {
        "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_pm1": train_metrics["acc_pm1"],
        "train_pm2": train_metrics["acc_pm2"],
        "train_mae": train_metrics["mae"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_pm1": val_metrics["acc_pm1"],
        "val_pm2": val_metrics["acc_pm2"],
        "val_mae": val_metrics["mae"],
    }
    history.append(row)

    print(f"Train loss {row['train_loss']:.4f} | acc {row['train_acc']:.4f} | MAE {row['train_mae']:.3f}")
    print(f"Val   loss {row['val_loss']:.4f} | acc {row['val_acc']:.4f} | MAE {row['val_mae']:.3f} | ±1 {row['val_pm1']:.4f}")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "best_val_acc": best_val_acc,
        "vectorizers": {"char": char_vectorizer, "char_wb": char_wb_vectorizer, "word": word_vectorizer},
        "decade_to_idx": decade_to_idx,
        "idx_to_decade": idx_to_decade,
        "decades": decades,
        "config": {
            "INPUT_DIM": INPUT_DIM,
            "NUM_CLASSES": NUM_CLASSES,
            "HIDDEN_1": HIDDEN_1,
            "HIDDEN_2": HIDDEN_2,
            "HIDDEN_3": HIDDEN_3,
            "DROPOUT": DROPOUT,
        }
    }, LAST_MODEL_PATH)

    if val_metrics["acc"] > best_val_acc:
        best_val_acc = val_metrics["acc"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "best_val_acc": best_val_acc,
            "best_val_metrics": val_metrics,
            "vectorizers": {"char": char_vectorizer, "char_wb": char_wb_vectorizer, "word": word_vectorizer},
            "decade_to_idx": decade_to_idx,
            "idx_to_decade": idx_to_decade,
            "decades": decades,
            "config": {
                "INPUT_DIM": INPUT_DIM,
                "NUM_CLASSES": NUM_CLASSES,
                "HIDDEN_1": HIDDEN_1,
                "HIDDEN_2": HIDDEN_2,
                "HIDDEN_3": HIDDEN_3,
                "DROPOUT": DROPOUT,
            }
        }, BEST_MODEL_PATH)

        print("Nuevo mejor checkpoint guardado.")
    else:
        epochs_without_improvement += 1
        print(f"Sin mejora por {epochs_without_improvement} época(s).")

    clean_memory(verbose=False)

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping activado.")
        break

print("Mejor época:", best_epoch)
print("Mejor val accuracy:", best_val_acc)


## 6. Evaluación, matriz de confusión y submission normal

In [ ]:
# =========================
# Curvas, test, matriz de confusión y submission del modelo con validación
# =========================

history_df = pd.DataFrame(history)
display(history_df)

plt.figure(figsize=(10, 4))
plt.plot(history_df["epoch"], history_df["train_acc"], label="Train acc")
plt.plot(history_df["epoch"], history_df["val_acc"], label="Val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()
plt.grid(True)
plt.show()

# Cargar checkpoint en CPU para evitar OOM
checkpoint = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Mejor época:", checkpoint["epoch"])
print("Best val acc:", checkpoint["best_val_acc"])

test_metrics = evaluate(model, test_loader, X_test, desc="Test")
print("TEST:", test_metrics)

test_pred_idx, test_logits = predict_sparse(model, X_test)
y_true = y_test
y_pred = test_pred_idx

print("Accuracy test:", accuracy_score(y_true, y_pred))
print("MAE test:", mean_absolute_error([idx_to_decade[i] for i in y_true], [idx_to_decade[i] for i in y_pred]))

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
cm_df = pd.DataFrame(cm, index=[idx_to_decade[i] for i in range(NUM_CLASSES)], columns=[idx_to_decade[i] for i in range(NUM_CLASSES)])
display(cm_df)

plt.figure(figsize=(15, 12))
plt.imshow(cm, aspect="auto")
plt.title("Matriz de confusión sin normalizar")
plt.xlabel("Década predicha")
plt.ylabel("Década real")
plt.xticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)], rotation=90)
plt.yticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)])
plt.colorbar(label="Conteo")
plt.tight_layout()
plt.show()

print("Predicciones por clase en test:")
display(pd.Series([idx_to_decade[i] for i in y_pred]).value_counts().sort_index())

# Submission con modelo de validación
EVAL_PATH = "eval.csv"
SUBMISSION_PATH = "submission_validation_model.csv"

eval_df = pd.read_csv(EVAL_PATH)
assert "id" in eval_df.columns and "text" in eval_df.columns
eval_df["text"] = eval_df["text"].fillna("").astype(str)

vectorizers = checkpoint["vectorizers"]
X_eval = sp.hstack([
    vectorizers["char"].transform(eval_df["text"].tolist()),
    vectorizers["char_wb"].transform(eval_df["text"].tolist()),
    vectorizers["word"].transform(eval_df["text"].tolist()),
], format="csr", dtype=np.float32)

eval_pred_idx, _ = predict_sparse(model, X_eval)
answers = [idx_to_decade[int(i)] for i in eval_pred_idx]

submission = pd.DataFrame({"id": eval_df["id"].values, "answer": answers})
submission.to_csv(SUBMISSION_PATH, index=False)

print("Submission guardada:", SUBMISSION_PATH)
display(submission.head())
print("Predicciones por década en eval:")
display(submission["answer"].value_counts().sort_index())


## 7. Entrenamiento full-data full-batch y submission final

In [ ]:
# =========================
# Entrenamiento final full-data con full-batch y submission
# =========================

FINAL_EPOCHS = int(checkpoint["epoch"])
FINAL_CHECKPOINT_DIR = CHECKPOINT_DIR + "_final"
os.makedirs(FINAL_CHECKPOINT_DIR, exist_ok=True)
FINAL_MODEL_PATH = os.path.join(FINAL_CHECKPOINT_DIR, "final_full_data.pt")

print("Entrenando modelo final full-batch con todo train.csv por", FINAL_EPOCHS, "épocas.")

full_texts = df[TEXT_COL].tolist()
y_full = df["label_idx"].values.astype(np.int64)

final_char_vectorizer = TfidfVectorizer(
    analyzer="char", ngram_range=CHAR_NGRAM_RANGE, min_df=MIN_DF, max_df=MAX_DF,
    max_features=CHAR_MAX_FEATURES, sublinear_tf=True, lowercase=False,
    strip_accents=None, dtype=np.float32
)
final_char_wb_vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=CHAR_WB_NGRAM_RANGE, min_df=MIN_DF, max_df=MAX_DF,
    max_features=CHAR_WB_MAX_FEATURES, sublinear_tf=True, lowercase=False,
    strip_accents=None, dtype=np.float32
)
final_word_vectorizer = TfidfVectorizer(
    analyzer="word", ngram_range=WORD_NGRAM_RANGE, min_df=MIN_DF, max_df=MAX_DF,
    max_features=WORD_MAX_FEATURES, sublinear_tf=True, lowercase=False,
    strip_accents=None, token_pattern=r"(?u)\b\w+\b", dtype=np.float32
)

print("Ajustando TF-IDF final...")
X_full = sp.hstack([
    final_char_vectorizer.fit_transform(full_texts),
    final_char_wb_vectorizer.fit_transform(full_texts),
    final_word_vectorizer.fit_transform(full_texts),
], format="csr", dtype=np.float32)

FINAL_INPUT_DIM = X_full.shape[1]
print("X_full:", X_full.shape)

try:
    model.to("cpu")
    del model
except Exception:
    pass
clean_memory()

final_model = SparseTfidfMLP(FINAL_INPUT_DIM, NUM_CLASSES).to(DEVICE)
final_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
final_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(final_optimizer, T_max=max(FINAL_EPOCHS, 1))

def train_final_fullbatch_epoch(model, X_matrix, y_array):
    model.train()
    X_tensor = csr_batch_to_torch_sparse(X_matrix, device=DEVICE)
    labels = torch.tensor(y_array, dtype=torch.long, device=DEVICE)

    final_optimizer.zero_grad(set_to_none=True)
    logits = model(X_tensor)
    loss = final_criterion(logits, labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    final_optimizer.step()

    metrics = compute_metrics_from_logits(logits.detach(), labels.detach())
    metrics["loss"] = loss.item()

    del X_tensor, labels, logits, loss
    clean_memory(verbose=False)
    return metrics

final_history = []
for epoch in range(1, FINAL_EPOCHS + 1):
    print(f"Final epoch {epoch}/{FINAL_EPOCHS}")
    m = train_final_fullbatch_epoch(final_model, X_full, y_full)
    final_scheduler.step()
    final_history.append({"epoch": epoch, **m, "lr": final_optimizer.param_groups[0]["lr"]})
    print(m)

torch.save({
    "epoch": FINAL_EPOCHS,
    "model_state_dict": final_model.state_dict(),
    "history": final_history,
    "vectorizers": {"char": final_char_vectorizer, "char_wb": final_char_wb_vectorizer, "word": final_word_vectorizer},
    "decade_to_idx": decade_to_idx,
    "idx_to_decade": idx_to_decade,
    "decades": decades,
    "config": {"INPUT_DIM": FINAL_INPUT_DIM, "NUM_CLASSES": NUM_CLASSES, "HIDDEN_1": HIDDEN_1, "HIDDEN_2": HIDDEN_2, "HIDDEN_3": HIDDEN_3, "DROPOUT": DROPOUT}
}, FINAL_MODEL_PATH)

print("Modelo final guardado:", FINAL_MODEL_PATH)

FINAL_SUBMISSION_PATH = "submission_full_data_model.csv"
eval_df = pd.read_csv(EVAL_PATH)
eval_df["text"] = eval_df["text"].fillna("").astype(str)

X_eval_final = sp.hstack([
    final_char_vectorizer.transform(eval_df["text"].tolist()),
    final_char_wb_vectorizer.transform(eval_df["text"].tolist()),
    final_word_vectorizer.transform(eval_df["text"].tolist()),
], format="csr", dtype=np.float32)

final_preds, _ = predict_sparse(final_model, X_eval_final, batch_size=BATCH_SIZE)
answers = [idx_to_decade[int(i)] for i in final_preds]

submission_final = pd.DataFrame({"id": eval_df["id"].values, "answer": answers})
submission_final.to_csv(FINAL_SUBMISSION_PATH, index=False)

print("Submission full-data guardada:", FINAL_SUBMISSION_PATH)
display(submission_final.head())
display(submission_final["answer"].value_counts().sort_index())
